# 가림 증강은 실제 스내치 영상의 연속 미탐을 줄였을까?

베이스라인과 회색·배경 질감 가림 학습의 `best.pt`를 같은 원본 77프레임에서 비교합니다. 실제 부분 가림이 있는 `a03`의 시작 공백, 이후 상승 구간에서 잃거나 되찾은 프레임, 비교 영상 `a05`의 새 미탐을 확인합니다. 주 비교 신뢰도는 0.10이며 기존 0.25 결과도 같은 후보에서 집계합니다.

06번 노트북이 저장한 `per_frame.csv`에서 기준 프레임과 정답 좌표를 읽습니다. 정답은 기존 라벨 12개와 평가용 판독 65개입니다. 두 영상 모두 준비부터 받기까지를 평가합니다. GPU를 사용하지 않으며, 프로젝트 커널에서 위에서부터 실행하면 됩니다.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch, Rectangle

from wlcv.config import ANNOTATIONS, ROOT, RUNS, VIDEOS
from wlcv.data.frames import read_rgb
from wlcv.evaluation.temporal import evaluate_frames, compare_thresholds

video_ids = ["a03_sn_f_01", "a05_sn_f_01"]
models = {"Baseline": "plate_baseline", "Gray": "plate_occlusion", "Texture": "plate_occlusion_texture"}
thresholds = (0.10, 0.25)
review_csv = ANNOTATIONS / "temporal_review_boxes.csv"
report = RUNS / "plate_baseline/report"
comparison_dir = report / "temporal_model_comparison"
comparison_dir.mkdir(parents=True, exist_ok=True)

previous = pd.read_csv(report / "temporal_misses/per_frame.csv")
columns = ["video_id", "frame_index", "label_source", "x1", "y1", "x2", "y2"]
fixed_frames = previous.loc[previous.threshold == 0.10, columns].reset_index(drop=True)
display(fixed_frames.groupby(["video_id", "label_source"]).size().unstack())

## 1. 같은 정답과 추론 설정으로 모델 비교

원본 MP4를 한 프레임씩 읽습니다. 세 모델 모두 Mac CPU, 입력 크기 640, 후보 하한 0.001, 매칭 IoU 0.50을 사용합니다. 정답 박스와 프레임 번호가 기존 기록과 같은지 확인한 다음 결과를 합칩니다.

In [ ]:
frame_results, miss_runs, totals, model_boxes = [], [], [], []
for name, run in models.items():
    print(name)
    weights = RUNS / run / "weights/best.pt"
    frames, boxes = evaluate_frames(weights, review_csv, video_ids)
    pd.testing.assert_frame_equal(frames, fixed_frames, check_exact=False, rtol=0, atol=1e-8)
    detail, intervals, summary = compare_thresholds(frames, boxes, thresholds)
    frame_results.append(detail.assign(model=name))
    miss_runs.append(intervals.assign(model=name))
    totals.append(summary.assign(model=name))
    model_boxes.append(boxes.assign(model=name))

detail = pd.concat(frame_results, ignore_index=True)
intervals = pd.concat(miss_runs, ignore_index=True)
summary = pd.concat(totals, ignore_index=True)
candidates = pd.concat(model_boxes, ignore_index=True)
timing = pd.read_csv(ROOT / "data/metadata/video_timing.csv").set_index("video_id")
fps = summary.video_id.map(timing.fps_from_median_gap)
summary["longest_seconds"] = (summary.longest_miss / fps).round(3)

for name, table in [("summary", summary), ("miss_runs", intervals),
                    ("per_frame", detail), ("candidates", candidates)]:
    table.to_csv(comparison_dir / f"{name}.csv", index=False)
display(summary.sort_values(["threshold", "video_id"]))

## 2. 베이스라인이 놓쳤던 프레임과 새로 놓친 프레임

`recovered`는 베이스라인이 놓쳤지만 해당 모델이 잡은 장수이고, `new_misses`는 베이스라인이 잡았지만 해당 모델이 놓친 장수입니다. 두 값을 따로 봐야 앞에서 잡은 한 장과 뒤에서 놓친 여러 장을 구분할 수 있습니다.

In [ ]:
paired = detail.pivot(index=["threshold", "video_id", "frame_index"], columns="model", values="detected")
changes = []
for name in ["Gray", "Texture"]:
    outcomes = pd.DataFrame({
        "recovered": paired[name] & ~paired["Baseline"],
        "new_misses": paired["Baseline"] & ~paired[name],
    })
    counts = outcomes.groupby(level=["threshold", "video_id"]).sum()
    changes.append(counts.assign(model=name).reset_index())
changes = pd.concat(changes, ignore_index=True)
changes.to_csv(comparison_dir / "paired_changes.csv", index=False)
display(changes.sort_values(["threshold", "video_id"]))

## 3. 연속 미탐의 위치

한 칸이 원본 영상 한 프레임입니다. 밝은 칸은 정탐, 진한 칸은 미탐입니다. 같은 열에서 세 모델을 비교하고, 같은 행에서는 미탐이 몇 장씩 이어지는지 확인하세요.

In [ ]:
colors = ["#93391f", "#cae8e4"]
fig, axes = plt.subplots(2, 2, figsize=(12, 4.6), layout="constrained")
for row, threshold in enumerate(thresholds):
    for col, video in enumerate(video_ids):
        axis = axes[row, col]
        grid = paired.xs((threshold, video), level=("threshold", "video_id"))
        first, last = grid.index.min(), grid.index.max()
        axis.imshow(grid[list(models)].to_numpy().T, cmap=ListedColormap(colors),
                    vmin=0, vmax=1, interpolation="nearest", aspect="auto",
                    extent=(first - 0.5, last + 0.5, 2.5, -0.5))
        axis.set_yticks(range(3), list(models))
        ticks = [tick for tick in range(5, last, 5) if first + 2 <= tick <= last - 2]
        axis.set_xticks([first, *ticks, last])
        for boundary in [0.5, 1.5]:
            axis.axhline(boundary, color="white", linewidth=0.8)
        axis.set_title(f"{video}   |   confidence ≥ {threshold:.2f}")
        axis.set_xlabel("Frame (0-based)")
fig.legend(handles=[Patch(color=colors[0], label="Missed"), Patch(color=colors[1], label="Detected")],
           loc="outside upper center", ncol=2, frameon=False)
fig.savefig(comparison_dir / "detection_timeline.png", dpi=150)
plt.show()

## 4. 결과가 엇갈린 실제 프레임

`a03`의 14번(회색 모델이 복구), 25번(두 증강 모델의 새 미탐), 31번(질감 모델이 복구)을 봅니다. 실선은 정답, 점선은 정답과 IoU 0.50 이상인 후보 중 신뢰도가 가장 높은 박스입니다. 후보 점수가 0.10을 넘는지 박스 위치와 함께 확인하세요.

In [ ]:
video = "a03_sn_f_01"
examples = [14, 25, 31]
fig, axes = plt.subplots(len(examples), len(models), figsize=(9, 7.4), layout="constrained")
for row, frame in enumerate(examples):
    rgb = read_rgb(VIDEOS / f"{video}.mp4", frame)
    truth = fixed_frames[(fixed_frames.video_id == video) & (fixed_frames.frame_index == frame)].iloc[0]
    in_frame = candidates[(candidates.video_id == video) & (candidates.frame_index == frame)]
    for axis, name in zip(axes[row], models):
        axis.imshow(rgb)
        axis.add_patch(Rectangle((truth.x1, truth.y1), truth.x2 - truth.x1, truth.y2 - truth.y1,
                                 fill=False, edgecolor="#46deff", linewidth=2))
        matches = in_frame[(in_frame.model == name) & (in_frame.iou >= 0.50)]
        label = "No match"
        if not matches.empty:
            box = matches.loc[matches.confidence.idxmax()]
            axis.add_patch(Rectangle((box.x1, box.y1), box.x2 - box.x1, box.y2 - box.y1,
                                     fill=False, edgecolor="#ffad40", linewidth=2, linestyle="--"))
            label = f"conf {box.confidence:.3f} · IoU {box.iou:.2f}"
        axis.set_title(f"f{frame} · {name}   |   {label}", fontsize=9)
        axis.set_xlim(max(0, truth.x1 - 60), min(rgb.shape[1], truth.x2 + 60))
        axis.set_ylim(min(rgb.shape[0], truth.y2 + 50), max(0, truth.y1 - 50))
        axis.axis("off")
fig.savefig(comparison_dir / "a03_model_boxes.png", dpi=150)
plt.show()